# Протоколы взаимодействия AI-агентов

**Тращенков Сергей · МФТИ · «Разработка и внедрение AI-агентов»**

Соберём помощника для поездки: он ищет поезда на Туту, подбирает продукты во ВкусВилле
и показывает найденные варианты пользователю. На этом примере разберём несколько протоколов:
что каждый из них передаёт, какие программы участвуют в обмене и как прочитать сообщения.

**Разделы 1–8 — MCP:** подключение инструментов, чтение ресурсов, получение шаблонов промптов,
собственный сервер и изменения спецификации и Python-библиотек.
**Разделы 9–12 — A2A:** обращение к другому агенту; два варианта сервера для одного агента и общий клиент.
**Разделы 13–15 — взаимодействие с приложением:** AG-UI, A2UI и Agent Client Protocol.
Далее — обзор протоколов покупок и оплаты, домашнее задание и дополнительный раздел о Skills.

Вспомогательный код собран в [`lab.py`](lab.py): [справочник по функциям](#lab-reference).
Его реализация показана в раскрываемых блоках рядом с соответствующими экспериментами.

Установка описана в [README](README.md). Перед выполнением ячеек заполните `.env` и начните с настройки в § 1.
Ячейки обращаются к реальным API: ответы модели, цены и результаты поиска могут меняться.
Выполняйте ячейки по порядку; модельные запросы могут расходовать средства на вашем API-аккаунте.

## 1. Участники взаимодействия и настройка

Пользователь просит найти поезд и подобрать продукты в дорогу. Для этого приложению нужны поиск,
логика выбора и способ показать ответ. Протоколы описывают разные виды обмена:

| Протокол | Для чего нужен | Пример в блокноте |
|---|---|---|
| MCP | Подключать инструменты и получать данные и шаблоны сообщений | Вызвать поиск по городам и дате, получить список поездов |
| A2A | Поручать задачу другому агенту и получать сведения о её выполнении | Отправить агенту поездок просьбу найти подходящий поезд |
| AG-UI | Передавать приложению события выполнения задачи | Сообщить, что агент начал поиск, и передавать ответ по частям |
| A2UI | Передавать описание элементов интерфейса | Описать карточку поезда с временем и ценой |
| Agent Client Protocol (ACP) | Управлять сессией агента и разрешениями на действия | Поручить исправление кода, подтвердить правку и запуск тестов |

Начнём с MCP. У него три участника:

- **Хост (приложение)** использует модель и подключает MCP-серверы. Примеры: Claude Desktop, Claude Code,
  редактор VS Code с поддержкой MCP или собственное Python-приложение. В наших опытах эту роль выполняет код блокнота.
- **MCP-клиент** — компонент приложения для обмена сообщениями с конкретным MCP-сервером.
  Для подключения двух серверов приложение создаёт два клиента.
- **MCP-сервер** — программа, которая предоставляет инструменты, данные и шаблоны сообщений.
  Например, сервер Туту предоставляет инструмент поиска поездов.

Агент — часть нашего приложения, которая решает пользовательскую задачу с помощью модели и инструментов.
При поиске поездки модель выбирает инструмент и аргументы. Код приложения отправляет этот вызов
через MCP-клиент серверу Туту. Сервер возвращает найденные поезда, приложение передаёт эти данные агенту,
и агент составляет ответ пользователю.

```text
Хост (приложение)
┌─────────────────────────────────────────┐
│ Агент ←→ код выполнения ←→ MCP-клиент    │ ← MCP → MCP-сервер
└─────────────────────────────────────────┘
```

MCP-клиент можно использовать и без агента: программа сама задаёт, какой инструмент вызвать или какой ресурс прочитать.
Так мы изучим возможности собственного сервера в § 5.1.

### Подготовка примера

Следующая ячейка импортирует библиотеки и функции запуска серверов, проверяет наличие ключа GigaChat
и задаёт дату поездки через 21 день. Выполните её перед остальными примерами.
Настройки из `.env` загружаются при импорте модуля `lab`.
`ROOT` — папка, в которой находится `lab.py`; пути к данным и серверам отсчитываются от неё.
`future_date(21)` возвращает дату через 21 день в формате `YYYY-MM-DD`.
`make_model()` создаёт клиент GigaChat; запрос к модели выполняется позже, при запуске агента.
Посмотрите настройки клиента, которые одинаковы для наших примеров.


<details>
<summary>Код из lab.py: make_model</summary>

```python
def make_model(**kwargs):
    return GigaChat(
        use_api_v2=True, model=os.getenv("GIGACHAT_MODEL", "GigaChat-3-Ultra"),
        temperature=0.01, timeout=120, **kwargs,
    )
```

</details>

In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path
from collections import Counter

import httpx
from IPython.display import display, HTML, Markdown
from langchain.mcp import MCPAdapter
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from deepagents import create_deep_agent

import lab
from lab import ROOT, make_model

TRIP_DATE = lab.future_date(21)
QUESTION = f"Найди самый дешёвый прямой поезд Москва — Казань на {TRIP_DATE}. Укажи номер, отправление, прибытие и цену. Только поиск."
print("Python:", sys.version.split()[0])
print("Дата поездки:", TRIP_DATE)
print("Модель:", os.getenv("GIGACHAT_MODEL", "GigaChat-3-Ultra"))
print("Ключ задан:", bool(os.getenv("GIGACHAT_CREDENTIALS")))
assert os.getenv("GIGACHAT_CREDENTIALS"), "Заполните GIGACHAT_CREDENTIALS в .env и повторите ячейку"

## 2. Инструменты, ресурсы и промпты MCP

**Кто предложил и развивает.** MCP предложила **Anthropic**. Сейчас проект размещён в
**Agentic AI Foundation (AAIF) при Linux Foundation**; техническим развитием занимаются сопровождающие проекта
и открытое сообщество. Разработчик конкретного MCP-сервера или SDK может быть другой организацией.
Источники: [передача проекта в AAIF](https://www.anthropic.com/news/donating-the-model-context-protocol-and-establishing-of-the-agentic-ai-foundation),
[управление проектом MCP](https://modelcontextprotocol.io/community/governance).

MCP-сервер предоставляет приложению три основных вида возможностей:

| Возможность | Что получает приложение | Пример |
|---|---|---|
| **Инструмент (tool)** | Описание операции и её аргументов; после вызова — данные или сообщение об ошибке | Передать города и дату в поиск, получить список поездов |
| **Ресурс (resource)** | Данные по заданному адресу — URI | Прочитать описание курса по `course://overview` |
| **Промпт (prompt)** | Подготовленные сообщения по шаблону с параметрами | Получить текст задания «Подготовь справку по MCP для студентов» |

**Инструмент выполняет операцию.** Для вызова нужно знать его имя и передать аргументы по описанной сервером схеме.
Агент может выбрать вызов с помощью модели; обычная программа может вызвать тот же инструмент напрямую.

**Ресурс предоставляет данные.** URI — идентификатор ресурса. Приложение читает содержимое через MCP-клиент
и решает, включать ли его в запрос к модели.

**Промпт предоставляет шаблон сообщений.** Приложение передаёт параметры, например тему и аудиторию,
а сервер возвращает готовые сообщения. Получение промпта не запускает модель: её вызывает само приложение.

В § 5–6 прочитаем все три каталога возможностей и используем их для подготовки справки по учебным материалам.

## 3. Изменения спецификации, Python SDK и LangChain

Спецификация задаёт правила обмена MCP. Её редакции обозначаются датой.
Python SDK — библиотека для реализации этих правил; у неё собственный номер версии.
Сначала сравним редакции протокола, затем изменения Python API.

### 3.1. Что изменилось в сообщениях MCP

| Что сравниваем | Прежний вариант (2025-11-25) | Используемый вариант (2026-07-28) |
|---|---|---|
| Начало обмена | Клиент отправляет `initialize`: сообщает свою версию и возможности, получает сведения о сервере | Версия и возможности клиента передаются в служебном поле `_meta` каждого запроса; сведения о сервере можно получить через `server/discover` |
| Состояние между вызовами | HTTP-сессия может иметь идентификатор `Mcp-Session-Id` | Сессии протокола нет. Если операции связаны, сервер возвращает идентификатор объекта, а клиент передаёт его в аргументах следующего вызова |

Например, чтобы продолжить обработку ранее созданного объекта, клиент передаёт выданный сервером идентификатор.
Само HTTP-соединение не определяет, с каким объектом работает приложение.
В § 7 посмотрим сообщения двух редакций на одном учебном сервере.

### 3.2. Если серверу не хватает данных

Допустим, приложение вызвало инструмент, но серверу нужно уточнить условие у пользователя.
Вместо окончательного ответа сервер возвращает `resultType: "input_required"` — «нужны дополнительные данные».
В поле `inputRequests` он описывает, что необходимо получить. Приложение показывает вопрос,
получает ответ пользователя и повторяет исходный запрос, добавив ответы в `inputResponses`.
После этого сервер продолжает обработку.

Такой обмен в несколько обращений называется **MRTR — Multi Round-Trip Requests**.
Уточнение данных у пользователя — один из случаев его применения.
В отличие от обычного завершённого результата (`resultType: "complete"`), `input_required` означает,
что обработка запроса ещё не закончена. Работа с таким ответом требует поддержки со стороны приложения.

### 3.3. Уведомления об изменениях и длительные операции

**Подписки** нужны, когда клиент хочет узнавать об изменениях сервера без постоянного повторного опроса.
Например, сервер может сообщить, что список инструментов обновился или изменилось содержимое ресурса.
Для этого клиент отправляет `subscriptions/listen`, указывает интересующие виды изменений и читает поток уведомлений.
Имя метода означает «слушать подписки». Уведомление об изменении не заменяет получение новых данных:
приложение при необходимости заново читает каталог или ресурс.

**Длительные задачи (Tasks)** позволяют начать операцию, получить её идентификатор и позже проверить состояние.
Например, подготовка большого отчёта может занять дольше обычного вызова инструмента.
В редакции 2026-07-28 этот механизм оформлен как отдельное расширение MCP:
он доступен, только если его поддерживают и клиент, и сервер. В наших ячейках это расширение не используется.

Некоторые дополнительные механизмы прежних редакций помечены устаревающими:

| Механизм | Для чего применялся | Статус в редакции 2026-07-28 |
|---|---|---|
| Roots | Клиент сообщал серверу корневые папки для работы | Устаревающий; в новых реализациях пути передают настройками сервера или аргументами операций |
| Sampling | Сервер просил приложение вызвать модель | Устаревающий; в новых реализациях вызов модели организуют напрямую |
| Logging | Сервер передавал диагностические сообщения средствами MCP | Устаревающий; для диагностики можно использовать журнал процесса |

«Устаревающий» означает, что механизм ещё присутствует в спецификации, но запланирован к удалению.
Основные примеры ниже используют инструменты, ресурсы и промпты.
Транспорты stdio и Streamable HTTP сравним в § 5.

### 3.4. Что изменилось в Python-коде

| Библиотека | Прежний API | API в этом блокноте |
|---|---|---|
| Python SDK MCP: сервер | SDK v1: класс `FastMCP` | SDK v2: `MCPServer`, настройка транспорта в `run()` |
| Python SDK MCP: клиент | Открыть транспорт, создать `ClientSession`, вызвать `await session.initialize()` | `async with Client(url)` для HTTP или `Client(params)` для stdio: клиент сам открывает подключение и выбирает совместимую редакцию протокола |
| Интеграция LangChain | Пакет `langchain-mcp-adapters`, класс `MultiServerMCPClient` | Дополнение `langchain[mcp]`, класс `MCPAdapter`, подключение через `async with`, получение инструментов через `list_tools()` |

`ClientSession` в SDK v2 остаётся доступным как низкоуровневый API; для обычных подключений используем `Client`.

`async with` ограничивает время подключения: внутри блока используем сервер, при выходе соединение освобождается.
`await` в ячейках означает ожидание завершения асинхронной операции, например ответа сервера.

Отдельная библиотека **FastMCP** также предоставляет клиентские и серверные средства; её использует адаптер LangChain.
Она имеет собственные версии. Одноимённый класс из прежнего официального SDK и отдельный пакет — разные вещи.

Интеграция `langchain.mcp` имеет статус beta. `MCPAdapter` превращает описания MCP-инструментов
в объекты инструментов LangChain. Ресурсы и промпты в наших примерах читаем через официальный Python SDK MCP.

Источники: [изменения спецификации MCP](https://modelcontextprotocol.io/specification/2026-07-28/changelog),
[Python SDK v2](https://py.sdk.modelcontextprotocol.io/whats-new/),
[миграция интеграции LangChain](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters).

## 4. Туту: сначала обнаруживаем возможности

Подключаем публичный сервер Туту по HTTP. `MCPAdapter` — клиентская обёртка в нашем приложении;
она преобразует описания серверных tools в инструменты LangChain. Подключение остаётся открытым, пока агент использует инструменты сервера.

В следующей ячейке модель не используется: получаем только каталог. Число и состав инструментов у внешнего сервиса
могут измениться. Посмотрите, какие задачи обещают решить описания и какие параметры потребуются для их выполнения.

In [ ]:
async with MCPAdapter("https://mcp.tutu.ru/mcp") as adapter:
    tutu_tools = await adapter.list_tools()
    for t in tutu_tools:
        print(t.name, "—", t.description.strip().splitlines()[0][:130])
    rail_tool = next(t for t in tutu_tools if t.name.endswith("search_rail"))
    rail_schema = rail_tool.args_schema
print("Инструментов:", len(tutu_tools))

### 4.1. Схема инструмента и необязательные параметры

JSON Schema задаёт структуру аргументов. Поле `required` сообщает, какие свойства обязательны.
`anyOf` разрешает несколько вариантов значения: например, целое число либо `null`.
Поле может быть обязательным в `required` и при этом допускать `null`: ключ должен присутствовать, но значение может быть пустым.
Схема поступает от внешнего MCP-сервера, поэтому совместимость нужно проверять по всей цепочке:
MCP → адаптер LangChain → клиент модели → API модели.

В выбранных версиях GigaChat API v2 работает с такими схемами; параметр `use_api_v2=True` выставляется в `make_model()`.
Команда `uv sync` устанавливает версии из `uv.lock`.
Далее используем исходные схемы, которые отдаёт сервер.

In [ ]:
schema = rail_schema if isinstance(rail_schema, dict) else rail_schema.model_json_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2)[:9000])

### 4.2. Где собирается и запускается агент

Список `tools` передаём в `create_deep_agent`. В `ainvoke` передаём сообщения пользователя.
Агент может сделать несколько шагов: вызвать инструмент, прочитать результат, уточнить поиск и сформировать ответ.
В Deep Agents этот цикл реализован графом LangGraph: вызов модели и выполнение инструментов — шаги,
между которыми передаётся история сообщений. Следующая ячейка создаёт агента, запускает его и печатает эту историю.

`recursion_limit` ограничивает число шагов графа. Это техническая граница выполнения, достижение которой не означает,
что агент успешно ответил. В учебном запуске после найденного результата просим завершить ответ.

In [ ]:
model = make_model()
try:
    async with MCPAdapter("https://mcp.tutu.ru/mcp") as adapter:
        tools = await adapter.list_tools()
        agent = create_deep_agent(
            model=model, tools=tools,
            system_prompt="Ищи сведения о поездке инструментами Туту. После успешного поиска дай краткий ответ и заверши работу. Не оформляй заказ.",
        )
        tutu_result = await agent.ainvoke(
            {"messages": [HumanMessage(content=QUESTION)]}, config={"recursion_limit": 60},
        )
finally:
    await model._client.aclose()
for message in tutu_result["messages"]:
    message.pretty_print()

### 4.3. Как читать историю агента в LangChain

Результат работы содержит историю сообщений. Здесь используются классы **LangChain**:

- `HumanMessage` — запрос пользователя.
- `AIMessage` — сообщение модели: текст ответа или выбранные вызовы инструментов с аргументами.
- `ToolMessage` — сообщение с результатом выполненного инструмента.

Это представление истории внутри нашего приложения. MCP между клиентом и сервером передаёт собственные
запросы и ответы; адаптер связывает этот обмен с инструментами LangChain.

Найдите в `AIMessage` вызов `search_rail` и сравните аргументы с запросом пользователя.
Затем найдите соответствующий `ToolMessage`: в нём находятся сведения о поездах, полученные от сервера.
Проверьте, что итоговый текст агента использует эти данные: города, дату, прямой маршрут, время и цену.
Цена и доступность относятся к моменту поиска.

**Вопрос:** что изменится, если пользователь не укажет дату? Когда агенту нужно задать уточняющий вопрос?

## 5. Собственный MCP-сервер

Сервер `servers/course_mcp.py` предоставляет доступ к учебным файлам из `data/course`.
В нём определены два ресурса, один шаблон промпта и два инструмента.
Откройте определения и найдите декораторы `@server.resource`, `@server.prompt` и `@server.tool`.
В `read_artifact_head` путь разрешается внутри папки данных; попытка выйти за её пределы отклоняется.

### Транспорты MCP

Транспорт определяет, как доставляются сообщения JSON-RPC между MCP-клиентом и сервером.
Для одного сервера состав инструментов, ресурсов и промптов может оставаться тем же при смене транспорта.
JSON-RPC — формат обмена: запрос содержит имя метода, аргументы и идентификатор `id`; ответ — тот же `id` и результат.
SSE (Server-Sent Events) позволяет серверу передавать сообщения потоком внутри HTTP-ответа.

| Транспорт | Как подключаемся | Что показываем |
|---|---|---|
| **stdio** | Хост через MCP-клиент запускает локальный дочерний процесс; обмен через stdin/stdout | В § 5.1 читаем списки возможностей, в § 6 используем их с агентом |
| **Streamable HTTP** | Клиент обращается по URL к отдельно запущенному серверу; ответ — JSON или SSE-поток | В § 7 запускаем тот же сервер по HTTP и смотрим запросы и ответы |

HTTP-сервер может работать на том же компьютере или удалённо. Старый транспорт **HTTP+SSE** заменён
Streamable HTTP; это не означает отказ от SSE-потоков внутри нового транспорта.
См. [архитектуру и транспорты MCP](https://modelcontextprotocol.io/docs/learn/architecture).

Сначала используем stdio: хост через MCP-клиент запускает Python-сервер как отдельный дочерний процесс.
stdin — поток ввода процесса сервера, stdout — его поток вывода. По ним передаются сообщения JSON-RPC.
Блок `async with` в Python-клиенте открывает подключение и завершает дочерний процесс после выхода из блока.
Сообщения протокола идут через stdout; диагностический вывод сервера направляют в stderr, чтобы не смешивать их.

In [ ]:
display(Markdown("```python\n" + (ROOT / "servers/course_mcp.py").read_text(encoding="utf-8") + "\n```"))

### 5.1. Три списка: tools, resources, prompts

Получим все три каталога через клиент официального Python SDK MCP, без агента и без обращения к модели.
В выводе найдите имена инструментов, адреса ресурсов и параметры промпта. Затем сравните прочитанный ресурс
с сообщениями, которые сервер подготовил по шаблону промпта.
После выхода из `async with` сервер stdio будет остановлен. Описания и прочитанные данные останутся в переменных ноутбука.

In [ ]:
from mcp import Client, StdioServerParameters

params = StdioServerParameters(command=sys.executable, args=[str(ROOT / "servers/course_mcp.py")])
async with Client(params) as client:
    listed_tools = await client.list_tools()
    listed_resources = await client.list_resources()
    listed_prompts = await client.list_prompts()
    print("TOOLS:", [t.name for t in listed_tools.tools])
    print("RESOURCES:", [str(r.uri) for r in listed_resources.resources])
    print("PROMPTS:", [p.name for p in listed_prompts.prompts])
    resource = await client.read_resource("course://overview")
    prepared = await client.get_prompt("prepare_lecture_brief", {"topic_name": "mcp", "audience": "студенты МФТИ"})
    overview_text = resource.contents[0].text
    prompt_texts = [m.content.text for m in prepared.messages]
print("RESOURCE:\n", overview_text)
print("PROMPT:\n", "\n".join(prompt_texts))

## 6. Собираем вход модели: resource + prompt + tools

Вход формирует приложение. Ресурс даёт описание курса, а промпт — сообщения с задачей подготовки справки.
Приложение объединяет их в список сообщений для модели.
Покажем весь получившийся контекст до запуска. Сам по себе URI ресурса не заставляет модель прочитать его.

Подключим tools через `MCPAdapter`. Во время ответа агент сможет запросить список материалов темы и прочитать начало файла.
Так в одном сценарии участвуют все три примитива: resource предоставляет исходный контекст, prompt задаёт структуру задачи,
а tools дают возможность получить подробности во время выполнения.

In [ ]:
brief_messages = [HumanMessage(content=t) for t in prompt_texts]
brief_messages.append(HumanMessage(content=f"Контекст курса из MCP-ресурса:\n{overview_text}\n\nПолучите список файлов темы mcp и прочитайте первые 6 строк первого файла. Затем подготовьте справку."))
for message in brief_messages:
    message.pretty_print()

In [ ]:
model = make_model()
try:
    async with MCPAdapter(ROOT / "servers/course_mcp.py") as adapter:
        course_agent = create_deep_agent(model=model, tools=await adapter.list_tools())
        brief = await course_agent.ainvoke({"messages": brief_messages}, config={"recursion_limit": 60})
finally:
    await model._client.aclose()
for message in brief["messages"]:
    message.pretty_print()

## 7. Тот же сервер по HTTP и сравнение редакций MCP

Включим у того же Python-сервера Streamable HTTP. Сервер будет работать отдельным процессом на порту 8765. Функция запуска сначала проверяет,
что он готов принимать запросы. Переменная `mcp_http` позволяет прочитать его журнал и остановить процесс.
После запуска сравним запрос новой редакции с подключением по прежним правилам.

`lab.start_server(...)` запускает дочерний процесс тем же Python, что использует ядро блокнота,
и ждёт ответа по адресу `ready_path`. Функция возвращает объект `LocalProcess`.
Его метод `read_log()` читает последние строки журнала из `.runtime/`, а `stop()` останавливает процесс.
Повторный запуск с тем же именем сначала останавливает предыдущий процесс этого примера.
Если порт занят другим процессом, функция сообщает об ошибке.
Это удобство запуска учебного стенда, а не часть MCP.


<details>
<summary>Код из lab.py: start_server</summary>

```python
def start_server(name, script=None, *, port, ready_path="/", args=(), agent_server=False):
    if old := SERVERS.pop(name, None):
        old.stop()
    with socket.socket() as sock:
        sock.settimeout(.3)
        if sock.connect_ex(("127.0.0.1", port)) == 0:
            raise RuntimeError(f"Порт {port} уже занят. Выберите другой порт или остановите свой сервер.")
    command = ([sys.executable, "-m", "langgraph_cli", "dev", "--no-reload", "--no-browser", "--port", str(port)]
               if agent_server else [sys.executable, "-m", str(Path(script).with_suffix("")).replace(os.sep, ".").replace("/", "."), *map(str, args)])
    server = LocalProcess(name, command)
    SERVERS[name] = server
    deadline = time.monotonic() + 90
    try:
        while time.monotonic() < deadline:
            if server.process.poll() is not None:
                raise RuntimeError(f"Сервер завершился:\n{server.read_log()}")
            try:
                response = httpx.get(f"http://127.0.0.1:{port}{ready_path}", timeout=1)
                if response.status_code < 500:
                    print(f"{name}: готов, порт {port}, PID {server.process.pid}")
                    return server
            except httpx.HTTPError:
                pass
            time.sleep(.2)
        raise TimeoutError(f"Сервер не готов:\n{server.read_log()}")
    except BaseException:
        server.stop()
        SERVERS.pop(name, None)
        raise
```

</details>

In [ ]:
mcp_http = lab.start_server("course-mcp", "servers/course_mcp.py", port=8765,
                            ready_path="/mcp", args=["--http", "--port", "8765"])

### 7.1. Запрос без предварительного initialize

Сначала получим сведения о сервере через `server/discover`, затем вызовем инструмент.
Найдите в запросе `_meta`: там указаны редакция протокола и возможности клиента.
`tools/call` означает вызов инструмента; имя инструмента и его аргументы передаются внутри запроса.
В ответе найдите `resultType`: значение `complete` означает, что получен окончательный результат.
Отдельный запрос `initialize` перед вызовом здесь не нужен.

`lab.mcp_request(method, params)` отправляет запрос на фиксированный адрес нашего сервера:
`http://127.0.0.1:8765/mcp`. Функция добавляет `jsonrpc`, уникальный `id`, версию и возможности клиента,
а также HTTP-заголовки. Параметр `request_meta=False` позволяет убрать новую служебную метаинформацию
для сравнения с прежним подключением в следующем опыте.

Возвращаются три значения: **отправленный запрос, ответ сервера и HTTP-заголовки ответа**.
`decode_json_response` разбирает обычный JSON-ответ или извлекает итоговое JSON-RPC-сообщение из SSE-потока.
Это небольшая функция для изучения сообщений, а не замена полноценному MCP-клиенту.


<details>
<summary>Код из lab.py: mcp_request</summary>

```python
async def mcp_request(method, params=None, *, version=PROTOCOL_VERSION, request_meta=True):
    params = dict(params or {})
    if request_meta:
        params["_meta"] = {"io.modelcontextprotocol/protocolVersion": version,
                           "io.modelcontextprotocol/clientCapabilities": {}}
    headers = {"Accept": "application/json, text/event-stream", "Mcp-Method": method,
               "MCP-Protocol-Version": version}
    if method == "tools/call":
        headers["Mcp-Name"] = params["name"]
    payload = {"jsonrpc": "2.0", "id": str(uuid4()), "method": method, "params": params}
    async with httpx.AsyncClient(timeout=30) as http:
        response = await http.post("http://127.0.0.1:8765/mcp", headers=headers, json=payload)
    return payload, decode_json_response(response), dict(response.headers)
```

</details>


<details>
<summary>Код из lab.py: decode_json_response</summary>

```python
def decode_json_response(response):
    response.raise_for_status()
    if "text/event-stream" in response.headers.get("content-type", ""):
        candidates = [json.loads(line[5:].strip()) for line in response.text.splitlines()
                      if line.startswith("data:") and line[5:].strip() != "[DONE]"]
        return next(item for item in reversed(candidates) if "result" in item or "error" in item)
    return response.json()
```

</details>

In [ ]:
request, discovery, headers = await lab.mcp_request("server/discover")
print("REQUEST:", json.dumps(request, ensure_ascii=False, indent=2))
print("RESPONSE:", json.dumps(discovery, ensure_ascii=False, indent=2))
request, called, headers = await lab.mcp_request("tools/call", {"name": "list_topic_artifacts", "arguments": {"topic_name": "mcp"}})
print("TOOL REQUEST:", json.dumps(request, ensure_ascii=False, indent=2))
print("TOOL RESPONSE:", json.dumps(called, ensure_ascii=False, indent=2))
assert "error" not in called, called

### 7.2. Подключение клиента прежней редакции

В прежней редакции обмен начинался с `initialize` — запроса согласования версии и возможностей.
Наш учебный сервер и SDK поддерживают этот способ подключения для совместимости. Отправим `initialize` с редакцией 2025-11-25.
Сравните ответ и HTTP-заголовки с предыдущим вызовом. Наличие совместимости на этом сервере не означает,
что любая пара старого и нового клиента будет поддерживать все возможности друг друга.

In [ ]:
old_request, old_response, old_headers = await lab.mcp_request(
    "initialize", {"protocolVersion": "2025-11-25", "capabilities": {}, "clientInfo": {"name": "mipt-demo", "version": "1.0"}},
    version="2025-11-25", request_meta=False,
)
print(json.dumps(old_response, ensure_ascii=False, indent=2))
print("Mcp-Session-Id:", old_headers.get("mcp-session-id", "не выдан"))
mcp_http.stop()

## 8. ВкусВилл и несколько MCP-серверов

Второй публичный сервер помогает подобрать продукты и создать ссылку на корзину.
Сначала посмотрим его каталог. Отдельные персональные возможности требуют авторизации пользователя.
Публичный поиск может показывать товары, доступность которых позже зависит от адреса доставки.

Для общего сценария используем конфигурацию с двумя серверами. Адаптер добавляет префиксы к именам инструментов,
чтобы приложение различало возможности разных серверов. Агент получает общий набор tools.

In [ ]:
connections = {"mcpServers": {
    "tutu": {"url": "https://mcp.tutu.ru/mcp"},
    "vkusvill": {"url": "https://mcp.vkusvill.ru/mcp"},
}}
async with MCPAdapter(connections) as adapter:
    combined_tools = await adapter.list_tools()
    print("Инструментов:", len(combined_tools))
    for t in combined_tools:
        print(t.name)

### 8.1. Один агент с инструментами двух серверов

Пользовательская задача содержит две части: поезд и перекус. В этой архитектуре один агент сам выбирает инструменты
обоих сервисов. По результату поиска продуктов он составляет аргументы создания ссылки на корзину.
Ссылка на корзину не означает оформление или оплату заказа.

Позже выполним ту же задачу с отдельным агентом по поездкам и сравним распределение работы.

In [ ]:
TRIP_AND_SNACK = (f"Еду {TRIP_DATE} из Москвы в Казань самым дешёвым прямым поездом. "
                  "Найди его и подбери во ВкусВилле перекус на одного из 3–4 продуктов. Дай ссылку на корзину. Не оформляй и не оплачивай заказ.")
model = make_model()
try:
    async with MCPAdapter(connections) as adapter:
        combined = create_deep_agent(model=model, tools=await adapter.list_tools(),
            system_prompt="Билеты ищи на Туту, продукты — во ВкусВилле. После поиска и получения ссылки дай итоговый ответ и заверши работу.")
        combined_result = await combined.ainvoke({"messages": [HumanMessage(content=TRIP_AND_SNACK)]}, config={"recursion_limit": 80})
finally:
    await model._client.aclose()
for message in combined_result["messages"]:
    message.pretty_print()

### 8.2. Доступ к личному кабинету: обзор OAuth

**Организация стандартизации.** OAuth 2.0 и PKCE — открытые стандарты **IETF**;
развитием семейства OAuth занимается рабочая группа OAuth. ВкусВилл использует эти механизмы,
а не является их автором. См. [рабочую группу IETF](https://datatracker.ietf.org/wg/oauth/about/) и
[стандарт PKCE](https://datatracker.ietf.org/doc/rfc7636/).

В персональном сценарии пользователь разрешает приложению доступ к определённым возможностям.
Например, доступ к истории заказов требует разрешения владельца личного кабинета.
Приложение направляет пользователя на страницу сервиса. После входа и согласия сервис выдаёт приложению
код авторизации, который оно обменивает на токен доступа. Токен предъявляют при обращении к защищённым операциям.
PKCE — проверка, которая связывает начало авторизации с обменом кода на токен и защищает от использования перехваченного кода.
Пользователь может отозвать выданное разрешение.

В этом блокноте используем публичные операции. Полный вход в личный кабинет не выполняется.
История заказов и персональные скидки не нужны для основного примера.
Описание реализации: [статья ВкусВилла об OAuth 2.0 + PKCE](https://habr.com/ru/companies/vkusvill/articles/1094732/).

**Практика:** добавьте в `course_mcp.py` инструмент подсчёта файлов в папке учебных материалов. Проверьте его наличие
через `list_tools()`, вызовите обычным клиентом и затем агентом. Сохраните ограничение доступа папкой `data/course`.

# Продолжение: взаимодействие агентов и приложений

## 9. A2A: задача для удалённого агента

**Кто предложил и развивает.** A2A предложила **Google**. Сейчас это открытый проект
**Agentic AI Foundation (AAIF) при Linux Foundation**, который развивают участники проекта из разных организаций.
Это общий протокол, а не интерфейс только к агентам Google.
Источник: [A2A в AAIF](https://aaif.io/blog/a2a-joins-aaif).

Теперь поручим поиск поезда отдельному агенту. **A2A (Agent-to-Agent)** задаёт обмен сообщениями
между приложением-клиентом и агентом, доступным по сети. Получив запрос, агент сам решает, как выполнить задачу.
В каждом обращении есть две стороны:

- **A2A-клиент** инициирует запрос. В § 10–11 эту роль выполняет код блокнота; в § 12 его использует агент-координатор.
- **A2A-сервер** принимает запрос и обеспечивает выполнение задачи агентом поездок. Он возвращает сообщения,
  состояние задачи и результаты.

Это роли в обмене, а не ограничение на число сообщений: клиент и сервер могут продолжать диалог,
например когда агенту нужно уточнение. В наших опытах клиент отправляет задачу и ждёт окончательный результат.

Клиенту нужны адрес агента, описание его возможностей и формат запроса. Эти сведения находятся в **Agent Card** — карточке агента.
Поле **`skills`** карточки описывает задачи, которые умеет решать агент: имя возможности, описание,
теги и примеры запросов. Например, `travel` — поиск поездки. Это сведения для выбора исполнителя;
они не перечисляют обязательно внутренние инструменты и не содержат инструкции `SKILL.md`.
В нашем клиенте нет отдельной команды «вызвать skill travel»: просьба о поездке отправляется обычным `SendMessage`.

Клиент обращается к публичному интерфейсу агента. Ему не нужно знать модель, внутренние инструменты,
MCP-серверы или подагентов исполнителя. Эти детали могут оставаться скрытыми.

A2A допускает два вида ответа: **Message** — сообщение, например непосредственный ответ или уточнение,
и **Task** — задача с состоянием выполнения. Результаты задачи передаются как **artifacts**:
это могут быть текст, структурированные данные или файлы. При поддержке потокового обмена клиент получает
обновления состояния и результатов по мере работы.

В нашем примере используется Task; обмен проходит так:

1. Клиент читает Agent Card и выбирает адрес для отправки сообщений.
2. Методом `SendMessage` отправляет просьбу найти поезд.
3. Сервер возвращает **Task** — объект задачи с идентификатором, состоянием и результатом работы.
4. Клиент проверяет состояние `completed` и читает **artifacts** — результаты задачи. Здесь это текст о найденной поездке.

Каждый запрос в следующих ячейках начинает новую задачу. В диалоге из нескольких обращений
идентификатор контекста позволяет связать сообщения с одним разговором.

Сравним **две реализации A2A-сервера для агента поездок**: собственный сервер на `a2a-sdk`
и встроенную поддержку A2A в LangGraph Agent Server. Каждый создаёт свой экземпляр агента через общую функцию
`agents.travel.travel_graph`. Модель, инструкции и инструменты одинаковы; клиент отправляет одинаковый запрос.
Так мы сравниваем серверную реализацию, сохраняя логику выполнения задачи.

In [ ]:
display(Markdown("```python\n" + (ROOT / "agents/travel.py").read_text(encoding="utf-8") + "\n```"))

## 10. Способ A: сервер на a2a-sdk

В `servers/a2a_sdk.py` явно определены Agent Card и `TravelExecutor`.
`TravelExecutor` — обработчик запроса: создаёт задачу, отмечает начало работы, запускает агента
и сохраняет его ответ как результат задачи. Агент реализован графом: шаги вызова модели и инструментов
выполняются до получения ответа. История этих шагов записывается в журнал сервера.
Задачи хранятся в памяти процесса: после его остановки они не сохраняются.

### Как запрос A2A попадает к агенту

**По протоколу:** A2A-сервер принимает сообщения клиента и возвращает ответы в формате A2A —
Message или Task, а при работе с задачей сообщает её состояние и результаты.
Протокол задаёт внешний обмен. Он не определяет, как внутри устроены агент, его модель,
инструменты и контекст, и не требует отдельного класса-обработчика.

**В нашей реализации на a2a-sdk:** `AgentExecutor` — класс Python SDK, связывающий обмен A2A
с агентом поездок на сервере. Обработчик извлекает текст через `context.get_user_input()`
и передаёт его этому агенту как сообщение с ролью `user`. После выполнения берёт ответ,
добавляет его в `artifacts` и отмечает завершение Task.

Агент поездок выполняет поиск с помощью модели и инструментов Туту. Обработкой протокола занимается
окружающий серверный код. Такое разделение позволяет использовать одну логику агента за разными интерфейсами.
Другая реализация может организовать это иначе. Передача текста как `user` и наличие отдельного
`AgentExecutor` не требуются спецификацией; роль `user` во внутреннем сообщении не означает,
что отправитель обязательно человек.

См. [роли участников A2A](https://a2a-protocol.org/latest/topics/key-concepts/) и
[пример AgentExecutor в Python SDK](https://a2a-protocol.org/latest/tutorials/python/4-agent-executor/).

`lab.discover_agent(base_url)` выполняет HTTP GET по пути `/.well-known/agent-card.json`
и возвращает карточку как Python-словарь. Получение карточки не запускает агента.


<details>
<summary>Код из lab.py: discover_agent</summary>

```python
async def discover_agent(base_url, *, assistant_id=None):
    async with httpx.AsyncClient(timeout=60) as http:
        response = await http.get(base_url.rstrip("/") + "/.well-known/agent-card.json",
                                  params={"assistant_id": assistant_id} if assistant_id else None)
        response.raise_for_status()
        return response.json()
```

</details>

In [ ]:
display(Markdown("```python\n" + (ROOT / "servers/a2a_sdk.py").read_text(encoding="utf-8") + "\n```"))
sdk_server = lab.start_server("a2a-sdk", "servers/a2a_sdk.py", port=9100, ready_path="/.well-known/agent-card.json")
sdk_card = await lab.discover_agent("http://127.0.0.1:9100")
print(json.dumps(sdk_card, ensure_ascii=False, indent=2))

### 10.1. Сообщение, задача и artifact

В небольшом JSON-RPC-клиенте видны сообщения протокола. Он использует адрес интерфейса из карточки.
В этом примере клиент ждёт, пока агент закончит работу, и получает окончательный ответ на `SendMessage`.
Рассмотрите отправленный запрос и статус задачи, затем прочитайте artifact.

`lab.ask_a2a(card, text)` выбирает из карточки JSON-RPC-интерфейс версии 1.0, создаёт сообщение пользователя
и отправляет `SendMessage` на адрес из карточки. Затем проверяет успешное завершение задачи и извлекает текст из `artifacts`.
Функция возвращает `A2AReply` с тремя полями: `request` — отправленный запрос, `task` — задача из ответа,
`text` — объединённый текст результатов. Эти поля печатаются следующей ячейкой.
Клиент рассчитан на наш пример: текстовый запрос, ожидание окончательного ответа и текстовые результаты.

### 10.2. Завершение задачи и качество результата

Состояние `TASK_STATE_COMPLETED` означает, что сервер сообщил о завершении задачи. Оно не подтверждает,
что найден действительно самый дешёвый поезд или соблюдены все условия пользователя.
`lab.ask_a2a` проверяет завершение и наличие текста, но не проверяет маршрут и достоверность цены.

Клиент может оценивать работу по проверяемому результату. Для поиска поездки полезно договориться
о структурированных полях: города, дата, пересадки, цена, ссылка на источник и время поиска.
Затем приложение проверяет обязательные поля и условия, а при необходимости сверяет сведения с источником.
Для сгенерированного кода проверкой могут быть тесты, для отчёта — обязательные разделы и ссылки.
Это требования приложения к результату, а не встроенная оценка качества A2A.

В выполняемом примере artifact содержит текст, и содержательную проверку проводим при чтении результата.
Убедительный текст и подробный журнал сами по себе не доказывают правильность ответа.


<details>
<summary>Код из lab.py: A2AReply</summary>

```python
@dataclass
class A2AReply:
    request: dict
    task: dict
    text: str
```

</details>


<details>
<summary>Код из lab.py: ask_a2a</summary>

```python
async def ask_a2a(card, text):
    interface = next(i for i in card["supportedInterfaces"]
                     if i["protocolBinding"] == "JSONRPC" and i["protocolVersion"] == "1.0")
    payload = {"jsonrpc": "2.0", "id": str(uuid4()), "method": "SendMessage", "params": {
        "message": {"messageId": str(uuid4()), "role": "ROLE_USER", "parts": [{"text": text}]}}}
    async with httpx.AsyncClient(timeout=300) as http:
        response = await http.post(interface["url"], headers={"A2A-Version": "1.0"}, json=payload)
        response.raise_for_status()
        envelope = response.json()
    if "error" in envelope:
        raise RuntimeError(f"A2A: {envelope['error']}")
    task = envelope["result"]["task"]
    if task["status"]["state"] != "TASK_STATE_COMPLETED":
        raise RuntimeError(f"Задача не завершилась успешно: {task['status']}")
    answer = "\n".join(p["text"] for a in task.get("artifacts", [])
                       for p in a.get("parts", []) if "text" in p)
    if not answer.strip():
        raise RuntimeError("Задача не вернула текстовый artifact")
    return A2AReply(payload, task, answer)
```

</details>

In [ ]:
sdk_reply = await lab.ask_a2a(sdk_card, QUESTION)
print("REQUEST:", json.dumps(sdk_reply.request, ensure_ascii=False, indent=2))
print("TASK:", sdk_reply.task["id"], sdk_reply.task["status"])
print("ARTIFACT:", sdk_reply.text)
print("\nИстория агента на сервере:\n", sdk_server.read_log(lines=70))

## 11. Способ B: встроенный A2A в LangGraph Agent Server

LangGraph Agent Server — готовый сервер для запуска агентов, построенных на LangGraph.
В `servers/agent_server.py` подключена та же функция создания агента, что и в первом варианте.
Файл `langgraph.json` задаёт путь к этому коду и имя `travel`, по которому сервер находит агента.

Команда `langgraph dev` запускает сервер локально. Сервер сам принимает A2A-запросы и управляет выполнением агента.
Нам не нужно заново писать обработчик `TravelExecutor`. Подключение к инструментам Туту остаётся открытым,
пока агент выполняет задачу.

Состояние агента сохраняет Agent Server. Поэтому при создании агента здесь не задаётся отдельный **checkpointer** —
компонент LangGraph для сохранения сообщений и промежуточных результатов между шагами.

`lab.platform_card()` сначала обращается к API **Agent Server**: ищет агента с `graph_id="travel"`
через `/assistants/search` и получает `assistant_id`. Затем читает A2A-карточку для этого агента.
Первый шаг нужен конкретному серверу LangGraph; общий A2A-клиент после получения карточки остаётся тем же.


<details>
<summary>Код из lab.py: platform_card</summary>

```python
async def platform_card(base_url="http://127.0.0.1:9300"):
    async with httpx.AsyncClient(timeout=60) as http:
        response = await http.post(base_url + "/assistants/search", json={"graph_id": "travel"})
        response.raise_for_status()
        assistant_id = response.json()[0]["assistant_id"]
    return await discover_agent(base_url, assistant_id=assistant_id)
```

</details>

In [ ]:
print((ROOT / "servers/agent_server.py").read_text(encoding="utf-8"))
print((ROOT / "langgraph.json").read_text(encoding="utf-8"))
platform_server = lab.start_server("agent-server", port=9300, ready_path="/ok", agent_server=True)
platform_card = await lab.platform_card()
print(json.dumps(platform_card, ensure_ascii=False, indent=2))

### 11.1. Тот же клиент и тот же запрос

Во втором запуске клиент читает карточку другого сервера и получает из неё новый адрес. Текст запроса, функция отправки и функция создания агента остаются прежними.
Внешний поиск выполняется дважды, поэтому цена и формулировки могут отличаться.
Сравните маршрут, дату, вызов инструмента и завершённый статус задачи.

In [ ]:
platform_reply = await lab.ask_a2a(platform_card, QUESTION)
print("TASK:", platform_reply.task["id"], platform_reply.task["status"])
print("ARTIFACT:", platform_reply.text)
print("\nОтвет сервера SDK:\n", sdk_reply.text)

In [ ]:
context_id = platform_reply.task["contextId"]
async with httpx.AsyncClient(timeout=30) as http:
    state_response = await http.get(f"http://127.0.0.1:9300/threads/{context_id}/state")
    state_response.raise_for_status()
    server_messages = state_response.json()["values"]["messages"]
print("Вызовы инструментов внутри Agent Server:", [call["name"] for m in server_messages for call in m.get("tool_calls", [])])

### 11.2. Сравнение серверных реализаций

| Слой | a2a-sdk | Agent Server |
|---|---|---|
| Логика агента | Общая функция создания Deep Agent с инструментами Туту | Та же функция |
| Agent Card и обработка запроса | Настраиваем в своём сервере | Предоставляет платформа |
| Задачи и состояние | `TaskUpdater` обновляет задачу, `InMemoryTaskStore` хранит её в памяти | Сохранением состояния и выполнением управляет Agent Server |
| Клиент основного примера | Общий JSON-RPC | Тот же JSON-RPC |

Клиент `lab.ask_a2a` получает адрес из Agent Card и отправляет JSON-RPC-запрос.
Найдите в ответе идентификатор задачи, `status.state` и текст в `artifacts`.
Сопоставьте эти поля у двух серверов: клиент использует их для чтения результата.

### Что доступно клиенту, а что мы смотрим отдельно

По A2A клиент получает опубликованные сообщения, состояние задачи и результаты.
Полная внутренняя история — вызовы модели, ответы инструментов и запуск субагентов — автоматически не передаётся.
Сервер может сообщать о ходе работы и включать подтверждающие данные в результаты, но не обязан раскрывать реализацию.

В блокноте дополнительно показаны журнал собственного сервера и запрос к `/threads/.../state` Agent Server.
Это отдельные средства наблюдения за нашим стендом. Координатор не получает эту историю через `lab.ask_a2a`.

Описание протокола: [участники и результаты A2A](https://a2a-protocol.org/latest/topics/key-concepts/),
[жизненный цикл задачи](https://a2a-protocol.org/latest/topics/life-of-a-task/).
Источники: [A2A endpoint](https://docs.langchain.com/langsmith/server-a2a),
[фабрики графов](https://docs.langchain.com/langsmith/graph-rebuild).

## 12. Координатор: A2A и MCP в одном приложении

В предыдущих разделах мы напрямую обращались к агенту поездок. Теперь добавим агента-координатора:
он получит общую просьбу о поездке и перекусе и распределит работу. В системе будет два агента:
координатор в блокноте — на клиентской стороне A2A, агент поездок — за A2A-сервером.
Координатор вызывает `travel_agent` с запросом на естественном языке. Внутри функции A2A-клиент отправляет
сообщение серверу поездок. В каждом запуске выбирается один из двух серверных вариантов — A или B.
Удалённый агент сам выбирает инструменты Туту. Продукты координатор ищет через MCP ВкусВилла.

Следующая функция одинакова для обоих серверов. Выбор карточки определяет, куда уйдёт запрос о поездке.
Координатор получает итог поиска поездки. Подробную историю вызовов модели и инструментов
удалённого агента смотрим в журнале его сервера.

### 12.1. Удалённый агент как действие координатора

Для координатора `travel_agent` — инструмент делегирования: он передаёт задачу отдельному агенту,
который сам решает, какие шаги выполнить. По роли в общей задаче это похоже на субагента.
Но A2A-исполнитель доступен по сети и может иметь собственные модель, состояние и инструменты.
Обёртка `@tool` — способ подключить его к нашему координатору на LangChain, а не требование A2A.

### 12.2. Ожидание и фоновое выполнение

В текущем коде `travel_agent` ожидает `await lab.ask_a2a(...)` и возвращает окончательный текст.
Во время ожидания цикл событий Python может выполнять другие асинхронные задачи, но этот вызов инструмента
ещё не завершён. `async def` сам по себе не организует продолжение планирования координатора.

A2A позволяет вернуть Task до завершения работы: в конфигурации `SendMessage` задаётся
`returnImmediately: true`. Клиент сохраняет идентификатор задачи и может продолжить другую работу.
Позже он получает состояние и результат через `GetTask`, подписку на обновления либо push-уведомления,
если соответствующий способ поддерживается сервером.

Для нашего сценария возможен такой порядок:

```text
Отправить задачу поиска поезда → получить taskId
Подобрать продукты во ВкусВилле
Получить результат поиска по taskId
Собрать общий ответ
```

Для этого нужны сервер с поддержкой выбранного режима, отдельные операции отправки и получения результата
и организация независимых действий координатора. Нынешний `lab.ask_a2a` рассчитан на завершённую Task
и воспринимает незавершённое состояние как ошибку. Поэтому приведённая фоновая схема объясняет возможность
протокола; в ячейках ниже выполняется вариант с ожиданием, а не отдельный опыт фонового запуска.

[Режимы SendMessage в спецификации](https://a2a-protocol.org/latest/specification/#322-sendmessageconfiguration).

In [ ]:
async def coordinate(target_card):
    @tool
    async def travel_agent(request: str) -> str:
        """Задать удалённому агенту вопрос о поездке: направление, дата, условия поиска."""
        reply = await lab.ask_a2a(target_card, request)
        print("A2A:", reply.task["status"]["state"])
        return reply.text

    model = make_model()
    try:
        async with MCPAdapter("https://mcp.vkusvill.ru/mcp") as adapter:
            coordinator = create_deep_agent(
                model=model, tools=[travel_agent, *await adapter.list_tools()],
                system_prompt="Билеты ищи через travel_agent, продукты — через ВкусВилл. Получив билетные сведения и ссылку на корзину, дай итоговый ответ. Заказ не оформляй.",
            )
            return await coordinator.ainvoke({"messages": [HumanMessage(content=TRIP_AND_SNACK)]}, config={"recursion_limit": 80})
    finally:
        await model._client.aclose()

In [ ]:
coordinator_sdk = await coordinate(sdk_card)
for message in coordinator_sdk["messages"]:
    message.pretty_print()

In [ ]:
coordinator_platform = await coordinate(platform_card)
for message in coordinator_platform["messages"]:
    message.pretty_print()

Сравните вызовы координатора в двух запусках: обращение к удалённому агенту, поиск продуктов и создание ссылки.
Сравните также первый вариант из раздела 8, где один агент имел прямые MCP-подключения к обоим сервисам.
Обсудите, какую часть задачи удобно поручить отдельному агенту и какие дополнительные расходы даёт ещё один модельный запуск.

In [ ]:
sdk_server.stop()
platform_server.stop()

## 13. AG-UI: события для пользовательского приложения

**Кто предложил и развивает.** AG-UI появился по инициативе **CopilotKit** в сотрудничестве с
**LangChain** и **CrewAI**. Его развивают CopilotKit и открытое сообщество проекта AG-UI.
Пакет `ag-ui-langgraph` — готовая интеграция этого проекта с LangGraph; формат событий не является
собственным форматом LangGraph. Источник: [описание AG-UI и интеграций](https://docs.ag-ui.com/introduction).

### Зачем нужен AG-UI

Агент может искать поезд, вызывать несколько инструментов и постепенно формировать ответ.
Пользовательскому приложению нужно показывать ход работы: что поиск начался, что пришла часть ответа,
когда всё закончено и произошла ли ошибка. Одного итогового текста для такого интерфейса недостаточно.

**AG-UI (Agent–User Interaction)** задаёт общий формат обмена между пользовательским приложением
и сервером агента: приложение отправляет сообщения и данные запуска, сервер передаёт события выполнения.
Клиенту не приходится разбирать собственные внутренние события каждого агентного фреймворка.
Для подключения конкретного фреймворка нужен совместимый адаптер; в нашем сервере он преобразует
события LangGraph в AG-UI. Благодаря общему формату интерфейс и агентный сервер можно развивать отдельно.

Например, после `RUN_STARTED` приложение включает индикатор, после `TOOL_CALL_START` показывает «Ищу поезд»,
а при каждом `TEXT_MESSAGE_CONTENT` дописывает полученный фрагмент ответа. `RUN_FINISHED` завершает индикатор.
Сам внешний вид — подписи, карточки и индикаторы — реализует приложение. AG-UI задаёт данные для его обновления.

### Как это показано в блокноте

Клиент отправляет HTTP POST, а сервер возвращает SSE-поток (Server-Sent Events):
последовательность сообщений в одном открытом HTTP-ответе. SSE доставляет поток;
AG-UI определяет типы событий и их поля. Клиент обрабатывает события по мере поступления.
[Описание назначения AG-UI](https://docs.ag-ui.com/introduction).

Запускаем FastAPI-приложение с агентом в отдельном процессе. Ноутбук играет роль клиента:
отправляет запрос и обрабатывает события по мере поступления. В выводе блокнота увидим события и текст ответа; по ним разберём, что могло бы показать приложение чата.
`lab.agui_events(port, text)` создаёт идентификаторы диалога (`threadId`) и запуска (`runId`),
помещает запрос пользователя в `messages` и отправляет HTTP POST на `/agent`.
Она читает строки SSE, разбирает содержимое `data:` как JSON и сохраняет события в список.
Фрагменты `TEXT_MESSAGE_CONTENT` сразу печатаются; полный список возвращается для последующего разбора.
Ошибка `RUN_ERROR` или отсутствие `RUN_FINISHED` приводит к исключению.
В коде ниже видны и тело запроса, и обработка входящих событий.


<details>
<summary>Код из lab.py: agui_events</summary>

```python
async def agui_events(port, text):
    payload = {"threadId": str(uuid4()), "runId": str(uuid4()), "state": {}, "tools": [],
               "context": [], "forwardedProps": {}, "messages": [
                   {"id": str(uuid4()), "role": "user", "content": text}]}
    events = []
    async with httpx.AsyncClient(timeout=240) as http:
        async with http.stream("POST", f"http://127.0.0.1:{port}/agent", json=payload) as response:
            response.raise_for_status()
            async for line in response.aiter_lines():
                if not line.startswith("data:"):
                    continue
                event = json.loads(line[5:].strip())
                events.append(event)
                kind = event["type"]
                if kind == "TEXT_MESSAGE_CONTENT":
                    print(event.get("delta", ""), end="", flush=True)
                elif kind in {"RUN_STARTED", "RUN_FINISHED", "RUN_ERROR", "TOOL_CALL_START", "TOOL_CALL_RESULT"}:
                    print(f"\n{kind}: {event.get('toolCallName', event.get('message', ''))}", flush=True)
                if kind == "RUN_ERROR":
                    raise RuntimeError(event.get("message", "Ошибка AG-UI"))
    if not any(e["type"] == "RUN_FINISHED" for e in events):
        raise RuntimeError("Поток завершился без RUN_FINISHED")
    return events
```

</details>

In [ ]:
agui_server = lab.start_server("agui", "servers/agui.py", port=9200, ready_path="/openapi.json", args=["--port", "9200"])
events = await lab.agui_events(9200, QUESTION)
print("\nЧисло событий каждого типа:", dict(Counter(e["type"] for e in events)))

### 13.1. Что приложение может показать по событиям

| Событие | Возможное действие приложения |
|---|---|
| `RUN_STARTED` | Показать начало работы |
| `TOOL_CALL_START`, `TOOL_CALL_ARGS` | Показать операцию и её параметры |
| `TOOL_CALL_RESULT` | Показать полученные сведения |
| `TEXT_MESSAGE_CONTENT` | Дописать очередной фрагмент текста |
| `STATE_SNAPSHOT` | Обновить представление состояния |
| `RUN_FINISHED`, `RUN_ERROR` | Завершить индикатор работы или показать ошибку |

Названия событий не определяют внешний вид приложения. Клиент выбирает, как визуализировать каждый тип.
Посмотрим одно событие вызова и одно событие результата. Большой ответ инструмента выводим отдельно от общего потока.

In [ ]:
for kind in ["TOOL_CALL_START", "TOOL_CALL_RESULT", "STATE_SNAPSHOT", "RUN_FINISHED"]:
    event = next((e for e in events if e["type"] == kind), None)
    print(kind, "\n", json.dumps(event, ensure_ascii=False, indent=2)[:2000], "\n")
agui_server.stop()

## 14. A2UI: описание интерфейса в результате агента

**Кто предложил и развивает.** A2UI предложила **Google**. Открытый проект развивается при участии
команд Google, **CopilotKit** и других участников сообщества. CopilotKit участвует и в AG-UI, и в экосистеме A2UI,
но назначения этих форматов различаются.
Источники: [анонс Google](https://developers.googleblog.com/en/introducing-a2ui-an-open-project-for-agent-driven-interfaces/),
[участники A2UI](https://a2ui.org/ecosystem/community/).

A2UI описывает компоненты интерфейса и связанные данные. Например, агент может вернуть карточку поезда с временем
и ценой. Рендерер клиента выбирает зарегистрированные компоненты и строит из них экран.
Каталог компонентов — список типов, которые умеет показывать приложение: например, Text, Card и Row.
JSON задаёт тексты и связи между элементами; приложение содержит код их отображения.
Так агент описывает карточку поезда, а клиент превращает её в видимый интерфейс.
AG-UI в этом опыте передаёт события выполнения, а инструмент генерации возвращает описание A2UI.

В этом примере используем формат A2UI **v0.9**, поддерживаемый установленным адаптером.
Поля JSON ниже относятся именно к этой редакции. Для формата v1.0 нужен рендерер с поддержкой v1.0.
В следующей ячейке используем тот же сервер, включив `--a2ui`.

`lab.a2ui_operations(events)` ищет событие `TOOL_CALL_RESULT`, разбирает его содержимое как JSON
и извлекает список `a2ui_operations`. Она не генерирует интерфейс, а достаёт описание, уже полученное от агента.
Если нужного результата нет, функция сообщает об ошибке.


<details>
<summary>Код из lab.py: a2ui_operations</summary>

```python
def a2ui_operations(events):
    for event in events:
        if event.get("type") == "TOOL_CALL_RESULT":
            try:
                payload = json.loads(event.get("content", ""))
            except (ValueError, TypeError):
                continue
            if isinstance(payload, dict) and "a2ui_operations" in payload:
                return payload["a2ui_operations"]
    raise ValueError("В потоке нет описания A2UI")
```

</details>

In [ ]:
a2ui_server = lab.start_server("a2ui", "servers/agui.py", port=9201, ready_path="/openapi.json", args=["--a2ui", "--port", "9201"])
a2ui_events = await lab.agui_events(9201, QUESTION + " Покажи один результат карточкой A2UI и заверши ответ.")
operations = lab.a2ui_operations(a2ui_events)
print("\nA2UI JSON:\n", json.dumps(operations, ensure_ascii=False, indent=2))

### 14.1. Как связаны JSON и изображение карточки

`createSurface` создаёт область интерфейса (surface), в которой приложение размещает компоненты. `updateComponents` задаёт компоненты с идентификаторами и связями.
Контейнер `root` содержит дочерние элементы; `Text` показывает текст, `Card` группирует сведения, `Row` располагает их рядом.
`updateDataModel` позволяет обновлять связанные данные без замены структуры компонентов.
Найдите в JSON корневой элемент и ссылки на его дочерние элементы. Сопоставьте Text со строкой текста,
а Card и Row — с группировкой и расположением сведений на полученной карточке.

Ниже — **учебный статический просмотр базовых компонентов v0.9** в HTML-выводе ноутбука.
Он показывает Text, Card, Column, Row и Divider. Кнопки не выполняют действия; List с шаблоном и связанные данные
требуют полноценного рендерера. Сравните фактический JSON с полученным представлением.

`lab.preview_a2ui(operations)` собирает компоненты из `updateComponents` по их идентификаторам,
начинает с `root` и рекурсивно преобразует дочерние элементы в HTML. Текст экранируется перед вставкой.
Функция возвращает строку HTML; `display(HTML(...))` показывает её в выводе ячейки.
В примере нет обработки нажатий и последующих обновлений интерфейса.


<details>
<summary>Код из lab.py: preview_a2ui</summary>

```python
def preview_a2ui(operations):
    """Учебный статический просмотр базовых компонентов A2UI v0.9."""
    components = {c["id"]: c for op in operations for c in op.get("updateComponents", {}).get("components", [])}
    def render(identifier, seen=frozenset()):
        if identifier in seen or identifier not in components:
            return ""
        node = components[identifier]
        seen = seen | {identifier}
        kind = node.get("component")
        if kind == "Text":
            value = node.get("text", "")
            text = html.escape(value if isinstance(value, str) else json.dumps(value, ensure_ascii=False))
            return f'<p style="margin:6px 0">{text}</p>'
        if kind == "Divider":
            return "<hr>"
        children = node.get("children", [node["child"]] if "child" in node else [])
        if not isinstance(children, list):
            return "<pre>List: " + html.escape(json.dumps(children, ensure_ascii=False)) + "</pre>"
        body = "".join(render(child, seen) for child in children)
        style = "display:flex;gap:18px;flex-wrap:wrap" if kind == "Row" else ""
        if kind == "Card":
            style += ";border:1px solid #7bbfa6;border-radius:14px;padding:20px;background:#f1fff8"
        return f'<div style="{style}">{body}</div>'
    return '<div style="color:#123;font:16px sans-serif;padding:16px">' + render("root") + "</div>"
```

</details>

In [ ]:
display(HTML(lab.preview_a2ui(operations)))
a2ui_server.stop()

Инструмент генерации A2UI подключается в `servers/agui.py`; его реализация находится в `compat/a2ui_tool.py`.
Вызов агента возвращает описание компонентов, а `lab.preview_a2ui` отображает их в блокноте.
Источники: [AG-UI](https://docs.ag-ui.com/), [A2UI](https://a2ui.org/), [пример A2UI](https://a2ui.org/quickstart/).

## 15. ACP: исправление кода под контролем приложения

**Кто предложил и развивает.** Agent Client Protocol запущен **Zed** совместно с **Google** и Gemini CLI.
К совместному развитию присоединилась **JetBrains**; протокол развивают Zed, JetBrains и участники открытого проекта.
Источники: [история запуска и участие JetBrains](https://zed.dev/blog/jetbrains-on-acp),
[документация ACP](https://agentclientprotocol.com/get-started/introduction).

### Что делаем и зачем здесь ACP

В небольшом Python-проекте рассчитывается стоимость перекуса. Функция получает пары «цена, количество»,
но ошибочно складывает только цены. Например, две позиции по 120 рублей и три по 80 должны стоить 480 рублей,
а функция возвращает 200. В репозитории есть исходный модуль и четыре теста.

Блокнот выполняет роль приложения для работы с coding-агентом:

1. Копирует учебный проект в отдельную рабочую папку и показывает исходное падение тестов.
2. Запускает процесс агента и создаёт ACP-сессию для этой папки.
3. Просит **проанализировать** ошибку без изменений.
4. В **той же сессии** просит исправить реализацию и запустить тесты.
5. Получает запросы разрешения перед правкой файла и выполнением команды. Пользователь блокнота решает, разрешить ли действие.
6. Независимо проверяет изменённый файл и тесты после разрешённого выполнения.

Именно такие действия нужны клиенту coding-агента: передать контекст проекта, продолжить разговор,
показать ход работы и получить контроль перед изменением файлов или запуском команд.
Вместо блокнота таким клиентом может быть редактор, терминальное или другое приложение.

### Какие приложения уже работают с ACP

Приложение должно иметь реализацию ACP-клиента: встроенную или установленную как расширение.
Примеры на ноябрь 2026 года:

| Приложение | Как подключается ACP-агент |
|---|---|
| **Zed** | Встроенная поддержка внешних агентов в Agent Panel; установка из ACP Registry или настройка своего агента |
| **IDE JetBrains** — например, PyCharm и IntelliJ IDEA | Через AI Assistant: выбор агента из реестра или настройка собственного в `acp.json` |
| **Visual Studio Code** | Через расширение, например ACP Client |
| **Neovim и Emacs** | Через плагины: например CodeCompanion для Neovim и agent-shell для Emacs |
| **Obsidian** | Через плагины, например Agent Client |
| **Chrome / веб-интерфейс** | Например, Chrome ACP: расширение или веб-клиент соединяется с прокси, который запускает ACP-агента |

Это примеры клиентских приложений, а не список агентов. Поддержка в редакторе или браузере
зависит от указанной интеграции. Обычная веб-страница не запускает локальный процесс через stdin/stdout:
в Chrome ACP обмен идёт через WebSocket до прокси и затем через stdio до агента.
ACP задаёт общение с агентом; доступ к страницам и действия в браузере требуют отдельных инструментов.

Источники: [каталог клиентов ACP](https://agentclientprotocol.com/get-started/clients),
[Zed](https://zed.dev/docs/ai/external-agents),
[JetBrains AI Assistant](https://www.jetbrains.com/help/ai-assistant/acp.html),
[Chrome ACP и схема подключения](https://github.com/Areo-Joe/chrome-acp).

### Что даёт SDK разработчику приложения

Для своего приложения можно взять официальный SDK: библиотеки для Python, TypeScript и других языков
поддерживают клиентскую и агентскую стороны. SDK предоставляет типы сообщений и средства обмена запросами,
ответами и уведомлениями. Разработчик приложения связывает их с интерфейсом: показывает текст и ход работы,
запрашивает разрешение пользователя и передаёт его решение агенту.
Файловые и терминальные возможности клиента необязательны и объявляются при согласовании возможностей.
См. [Python SDK](https://agentclientprotocol.com/libraries/python)
и [TypeScript SDK](https://agentclientprotocol.com/libraries/typescript).

В этом блокноте `lab.ACPClient` — наш минимальный учебный клиент, написанный вручную для просмотра обмена JSON-RPC.
Вывод ячейки заменяет окно чата, а `input()` — кнопки разрешения и отказа.
На стороне агента используется готовый адаптер `deepagents-acp`.

### Как устроен обмен

Клиент запускает `servers/acp_server.py` отдельным процессом и общается через stdin/stdout, без HTTP-порта.
В запросе `initialize` **клиент → агент** передаёт версию ACP и возможности клиента;
в ответе **агент → клиент** возвращает выбранную версию и свои возможности.
`session/new` создаёт сессию и передаёт рабочую папку `cwd`. Оба запроса `session/prompt` используют один `sessionId`.

Готовый `AgentServerACP` из `deepagents-acp` связывает Deep Agent с протоколом.
Агент сообщает о чтении, изменении файлов и выполнении команд уведомлениями `session/update`.
Перед `edit_file`, `write_file` и `execute` штатный механизм подтверждений Deep Agents приостанавливает работу,
а ACP-адаптер отправляет клиенту **`session/request_permission`**. Агент продолжает действие после ответа клиента.

В этом опыте файловые операции и команды исполняет агент через `LocalShellBackend` в указанной рабочей папке.
Файловые инструменты используют пути от корня проекта: `/trip_budget.py` означает файл в рабочей папке.
Команда unittest выполняется из этой папки. Клиент обрабатывает разрешения, но не реализует отдельные ACP-методы `fs/read_text_file` или `terminal/create`.
Рабочая папка помогает организовать проект, но LocalShellBackend не является изолированной песочницей:
команды работают с правами Python-процесса. Поэтому разрешения на экране нужно читать перед подтверждением.

`lab.ACPClient` различает ответы, уведомления и встречные запросы агента. Параметр `on_update` показывает события
по мере получения; `permission_handler` выбирает вариант ответа из предложенных агентом.
Без обработчика разрешений клиент отклоняет действие. Журнал решений сохраняется в `permission_log`.

При появлении `Разрешить это действие один раз? [y/N]:` введите `y` и нажмите Enter для разрешения
или просто Enter для отказа. Пока ввода нет, ячейка остаётся выполняющейся, а предложенное действие
приостановлено. Само ожидание ввода не запускает новые обращения к модели.


<details>
<summary>Код из lab.py: ACPClient</summary>

```python
class ACPClient:
    """Stdio-клиент: уведомления и решения по session/request_permission."""

    def __init__(self, *, permission_handler=None, on_update=None):
        if old := SERVERS.pop("acp", None):
            old.stop()
        self.server = LocalProcess("acp", [sys.executable, "-m", "servers.acp_server"], stdin=True)
        SERVERS["acp"] = self.server
        self.responses = queue.Queue()
        self.counter = 0
        self.permission_handler = permission_handler
        self.on_update = on_update
        self.permission_log = []
        def read():
            for line in self.server.process.stdout:
                try:
                    self.responses.put(json.loads(line))
                except ValueError:
                    self.responses.put({"invalid_stdout": safe_log(line)})
            self.responses.put(None)
        threading.Thread(target=read, daemon=True).start()

    def request(self, method, params, timeout=300):
        self.counter += 1
        request = {"jsonrpc": "2.0", "id": self.counter, "method": method, "params": params}
        self.server.process.stdin.write(json.dumps(request, ensure_ascii=False) + "\n")
        self.server.process.stdin.flush()
        updates = []
        deadline = time.monotonic() + timeout
        while True:
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                raise TimeoutError(f"ACP: истекло время ожидания {method}")
            try:
                message = self.responses.get(timeout=remaining)
            except queue.Empty as error:
                raise TimeoutError(f"ACP: истекло время ожидания {method}") from error
            if message is None:
                raise RuntimeError(f"ACP-процесс завершился:\n{self.server.read_log()}")
            # Запрос агента и запрос клиента имеют независимые пространства id.
            if "method" in message and "id" in message:
                reply = {"jsonrpc": "2.0", "id": message["id"]}
                if message["method"] == "session/request_permission":
                    permission = message["params"]
                    options = permission.get("options", [])
                    reject = next((o["optionId"] for o in options if o["kind"] == "reject_once"), None)
                    selected = self.permission_handler(permission) if self.permission_handler else reject
                    outcome = ({"outcome": "selected", "optionId": selected}
                               if selected in {o["optionId"] for o in options}
                               else {"outcome": "cancelled"})
                    reply["result"] = {"outcome": outcome}
                    self.permission_log.append({"request": permission, "outcome": outcome})
                else:
                    reply["error"] = {"code": -32601, "message": "Client method not implemented"}
                self.server.process.stdin.write(json.dumps(reply, ensure_ascii=False) + "\n")
                self.server.process.stdin.flush()
                continue
            if message.get("id") == self.counter:
                if "error" in message:
                    raise RuntimeError(f"ACP: {message['error']}")
                return request, message["result"], updates
            if "invalid_stdout" in message:
                raise RuntimeError(f"В stdout ACP попала диагностика: {message['invalid_stdout']}")
            updates.append(message)
            if self.on_update:
                self.on_update(message)

    def stop(self):
        self.server.stop()
        SERVERS.pop("acp", None)
```

</details>

In [ ]:
import shutil
import subprocess

acp_workspace = ROOT / ".runtime/acp-workspace"
acp_workspace.mkdir(parents=True, exist_ok=True)
for name in ["trip_budget.py", "test_trip_budget.py"]:
    shutil.copy2(ROOT / "data/acp_project" / name, acp_workspace / name)
acp_before = {name: (acp_workspace / name).read_bytes()
              for name in ["trip_budget.py", "test_trip_budget.py"]}
acp_baseline = subprocess.run([sys.executable, "-m", "unittest", "-v"], cwd=acp_workspace,
                             capture_output=True, text=True, encoding="utf-8")
print("До исправления тесты должны падать:\n", acp_baseline.stdout + acp_baseline.stderr)
assert acp_baseline.returncode != 0


def show_acp_update(message):
    update = message.get("params", {}).get("update", {})
    kind = update.get("sessionUpdate")
    if kind == "agent_message_chunk":
        print(update.get("content", {}).get("text", ""), end="", flush=True)
    elif kind in {"tool_call", "tool_call_update"}:
        print("\nACP:", kind, update.get("title", ""), update.get("status", ""), flush=True)


def decide_acp_permission(params):
    print("\nАгент запрашивает разрешение:")
    print(json.dumps(params["toolCall"], ensure_ascii=False, indent=2))
    answer = input("Разрешить это действие один раз? [y/N]: ").strip().lower()
    kind = "allow_once" if answer in {"y", "yes", "д", "да"} else "reject_once"
    return next(option["optionId"] for option in params["options"] if option["kind"] == kind)


acp_client = lab.ACPClient(permission_handler=decide_acp_permission, on_update=show_acp_update)
request, initialized, _ = acp_client.request("initialize", {
    "protocolVersion": 1, "clientCapabilities": {}, "clientInfo": {"name": "mipt-notebook", "version": "1.0"},
})
print("INITIALIZE:", json.dumps(initialized, ensure_ascii=False, indent=2))
_, session, _ = acp_client.request("session/new", {"cwd": str(acp_workspace), "mcpServers": []})
print("SESSION:", session)

### 15.1. Первый запрос: только анализ

Просим прочитать реализацию и тесты. Агент должен обнаружить, что функция не учитывает количество товаров.
Смотрим уведомления о чтении файлов и текст объяснения. После ответа обычным Python-кодом проверяем,
что оба файла остались неизменными. Изучение проекта само по себе не даёт разрешения на правку.

In [ ]:
try:
    request, diagnosis, acp_analysis_updates = acp_client.request("session/prompt", {
        "sessionId": session["sessionId"],
        "prompt": [{"type": "text", "text": (
            "Прочитай /trip_budget.py и /test_trip_budget.py. "
            "Объясни причину ошибки и предложи исправление. Пока ничего не меняй и не запускай команды."
        )}],
    })
    print("\nCOMPLETION:", diagnosis)
    assert diagnosis["stopReason"] == "end_turn"
    assert all((acp_workspace / name).read_bytes() == content for name, content in acp_before.items())
    print("Файлы после анализа не изменены")
except BaseException:
    acp_client.stop()
    raise

### 15.2. Продолжение сессии: исправление и проверка

Не создаём новую сессию: отправляем вторую просьбу с тем же `sessionId`.
Агент уже видел проект и получает команду применить исправление. Перед изменением файла и командой тестов
появится запрос разрешения с описанием действия и аргументами. Введите `y`, чтобы разрешить его один раз,
или нажмите Enter, чтобы отклонить. Для полного успешного опыта нужно разрешить правку и проверку.

Покажем различие между ответом агента и результатом работы: после `end_turn` отдельно запускаем тесты,
сравниваем исходный и изменённый код и проверяем, что агент не подменил тесты.
Если действие отклонено, блокнот не обходит отказ автоматическим запуском проверки.

In [ ]:
import difflib

try:
    acp_test_command = f'"{sys.executable}" -m unittest -v'
    request, completion, acp_fix_updates = acp_client.request("session/prompt", {
        "sessionId": session["sessionId"],
        "prompt": [{"type": "text", "text": (
            "Теперь исправь найденную ошибку в trip_budget.py. Тесты не меняй. "
            f"Выполни команду {acp_test_command}. "
            "Если тесты не проходят, исправь реализацию и проверь снова. "
            "В конце кратко объясни изменение и результат проверки."
        )}],
    })
    print("\nCOMPLETION:", completion)
    print("Запросов разрешения:", len(acp_client.permission_log))
    assert completion["stopReason"] == "end_turn"
    permitted = bool(acp_client.permission_log) and all(
        item["outcome"].get("optionId") in {
            option["optionId"] for option in item["request"]["options"] if option["kind"] == "allow_once"
        }
        for item in acp_client.permission_log
    )
    if permitted:
        verified = subprocess.run([sys.executable, "-m", "unittest", "-v"], cwd=acp_workspace,
                                  capture_output=True, text=True, encoding="utf-8")
        print("Независимая проверка:\n", verified.stdout + verified.stderr)
        assert verified.returncode == 0
        assert (acp_workspace / "test_trip_budget.py").read_bytes() == acp_before["test_trip_budget.py"]
        assert (acp_workspace / "trip_budget.py").read_bytes() != acp_before["trip_budget.py"]
        print("".join(difflib.unified_diff(
            acp_before["trip_budget.py"].decode("utf-8").splitlines(keepends=True),
            (acp_workspace / "trip_budget.py").read_text(encoding="utf-8").splitlines(keepends=True),
            fromfile="до", tofile="после",
        )))
        print("Четыре теста проходят, файл изменён, исходные тесты сохранены")
    else:
        print("Действие не было разрешено: автоматическую проверку не запускаем. Разберите журнал решений.")
finally:
    acp_client.stop()

### Что показал этот опыт

- `cwd` в `session/new` задаёт проект, с которым работает агент.
- Два `session/prompt` с одним `sessionId` продолжают один диалог.
- `session/update` позволяет приложению показывать чтение, правку и результат команд.
- `session/request_permission` даёт клиенту возможность разрешить или отклонить предложенное действие.
- `end_turn` завершает ответ, а правильность исправления проверяется файлами и тестами.

Общение с моделью и файловыми инструментами выполняет агентный процесс. Блокнот управляет им через ACP
и не вызывает его внутренний граф напрямую. После опыта процесс остановлен; изменённый проект остаётся в
`.runtime/acp-workspace`. Повторный запуск подготовительной ячейки восстанавливает исходные два учебных файла.
[Документация Deep Agents ACP](https://docs.langchain.com/oss/python/deepagents/acp).

## 16. Коммерция и платёжные протоколы — обзор

Агент, который нашёл товар, должен согласовать условия покупки, получить разрешение пользователя и передать
платёжные сведения соответствующим участникам. Разные протоколы покрывают разные части этого процесса.
В этом разделе выполняемых платёжных примеров нет.

### Кто предложил протоколы и кто их развивает

| Протокол | Инициаторы | Развитие проекта |
|---|---|---|
| Agentic Commerce Protocol (ACP) | OpenAI и Stripe | Спецификацию сопровождают OpenAI и Stripe, принимаются предложения сообщества ([репозиторий](https://github.com/agentic-commerce-protocol/agentic-commerce-protocol)) |
| Universal Commerce Protocol (UCP) | Google совместно с Shopify и другими отраслевыми партнёрами | Google, Shopify и другие участники открытого проекта ([участники UCP](https://ucp.dev/)) |
| Agent Payments Protocol (AP2) | Google при участии платёжной отрасли | Google передала AP2 в FIDO Alliance; дальнейшая стандартизация ведётся в рабочей группе Payments Technical Working Group ([FIDO Alliance](https://fidoalliance.org/fido-alliance-to-develop-standards-for-trusted-ai-agent-interactions/)) |
| x402 | Coinbase | x402 Foundation при Linux Foundation; в развитии участвуют Coinbase, Cloudflare и другие организации ([Linux Foundation](https://www.linuxfoundation.org/press/linux-foundation-announces-operational-launch-of-x402-foundation-to-standardize-internet-native-payments-for-ai-agents-and-applications)) |

Инициатор протокола и организация, в рамках которой он развивается, могут различаться.
Сведения об организациях приведены по состоянию на 12 ноября 2026 года.

### Для каких задач нужны эти протоколы

| Протокол | Обсуждаемый предмет | Пример сообщения или этапа |
|---|---|---|
| **Agentic Commerce Protocol, ACP** | Взаимодействие агента и продавца при оформлении заказа | Создание и уточнение оформления покупки |
| **Universal Commerce Protocol, UCP** | Жизненный цикл коммерции | Поиск → корзина → оформление → работа с заказом |
| **Agent Payments Protocol, AP2** | Проверяемое делегирование и разрешение оплаты | CheckoutMandate и PaymentMandate — подписанные документы, подтверждающие условия покупки и разрешение на оплату |
| **x402** | Оплата доступа к HTTP-ресурсу | HTTP-ответ 402 Payment Required с условиями, затем запрос с подтверждением оплаты |

Сравните два сценария: покупка набора продуктов и платный запрос к API.
В первом нужны позиции заказа, условия и разрешение на оплату. Во втором требуется связать доступ к ресурсу с оплатой запроса.
AP2 может применяться как расширение UCP для подтверждения полномочий и согласия на оплату.

Аббревиатура ACP встречается также у **Agent Client Protocol**, который мы только что использовали для общения с агентным процессом.
При обсуждении всегда уточняйте полное название.

Источники: [Agentic Commerce Protocol](https://www.agenticcommerce.dev/), [UCP](https://ucp.dev/),
[AP2](https://ap2-protocol.org/), [связь UCP и AP2](https://ucp.dev/documentation/ucp-and-ap2/), [x402](https://x402.org/).

## 17. Итог основного занятия

Мы подключили готовые MCP-серверы, добавили инструменты, ресурсы и промпты на свой сервер, собрали запрос к модели,
выполнили задачу одним агентом с двумя серверами и двумя агентами с A2A. Затем посмотрели события приложения,
описание интерфейса и общение с агентным процессом по stdio.

**Контрольные вопросы**

1. Кто читает MCP-resource и решает передать его модели?
2. Что возвращает MCP-prompt и чем его получение отличается от запуска модели?
3. Как координатор узнаёт адрес A2A-сервера? Где выполняется внутренний поиск?
4. Что меняется, если того же агента запустить через LangGraph Agent Server?
5. Какое событие AG-UI сообщает об окончании запуска?
6. Кто превращает JSON A2UI в компоненты интерфейса?
7. Как клиент ACP узнаёт о предложенной правке и как может её отклонить?

<details>
<summary>Ответы для самопроверки</summary>

1. Ресурс читает MCP-клиент; приложение решает включить его содержимое в запрос к модели.
2. MCP-prompt возвращает подготовленные сообщения. Для ответа нужно отдельно вызвать модель.
3. Адрес берётся из Agent Card. Поиск по Туту выполняет удалённый агент через свой MCP-клиент.
4. Agent Server принимает A2A-запросы и управляет выполнением и состоянием агента; отдельный обработчик на a2a-sdk не нужен.
5. Об успешном окончании сообщает RUN_FINISHED; об ошибке — RUN_ERROR.
6. Экран строит рендерер приложения по описанию компонентов A2UI.
7. Агент отправляет session/request_permission; клиент выбирает reject_once. Уведомления session/update показывают ход работы, но сами по себе не являются разрешением.

</details>

In [ ]:
lab.stop_all()
print("Локальные серверы остановлены")

# Домашнее задание: RAG и MCP

Полные условия и критерии оценки: [PDF задания](homework_mcp_knowledge_base.pdf).

Подготовьте собственный корпус документов, выполните разбиение, добавьте метаданные и постройте индекс.
Эмбеддинги документов и запросов вычисляйте локальной моделью. Используйте одну модель и одинаковую настройку
нормализации при индексации и поиске; для моделей с префиксами соблюдайте их правила для документов и запросов.
Подключите **готовый MCP-сервер** поиска к агенту на LangChain, LangGraph или Deep Agents.
Агент должен вызывать поиск и возвращать top-k фрагментов с `document_id`, `chunk_id` и `source`.
Проверьте поиск на **15–20 запросах**, включая вопросы без ответа в корпусе, и приложите ручную оценку.
Требования к корпусу, состав сдачи и веса критериев приведены в PDF.
Дополнительно по желанию: передайте найденные top-k фрагменты и исходный вопрос в LLM, получите ответ на вопрос со ссылками на источники.

В PDF приведены команды запуска Qdrant и пример подключения через `MCPAdapter` к Deep Agents.
Используйте указанную версию сервера и отдельное окружение: это позволяет воспроизвести конфигурацию примера.

# Дополнительный материал: Skills

## 18. Навык как пакет инструкций и файлов

**Кто предложил и развивает формат.** Формат **Agent Skills** первоначально разработала **Anthropic**
и опубликовала как открытый стандарт. Он развивается с участием открытого сообщества.
Это формат инструкций и файлов, а не сетевой протокол. `SkillsMiddleware` — реализация его поддержки в Deep Agents,
которую развивает **LangChain**. Источники: [Agent Skills](https://agentskills.io/home),
[поддержка в Deep Agents](https://docs.langchain.com/oss/python/deepagents/skills).

Skill предоставляет агенту процедуру выполнения задачи. `SKILL.md` содержит имя, описание и инструкции;
рядом могут находиться скрипты, шаблоны и справочные материалы. По описанию агент выбирает релевантный навык,
а полный текст читает, когда приступает к задаче. Это помогает загружать подробности по мере необходимости.

Для практики используем QR-навык: из заданной строки он создаёт PNG, затем читает QR-код обратно.
Результат можно проверить независимым декодером. В папке навыка находится `SKILL.md`, а в `scripts/` — `qr_generate.py` и `qr_read.py`.
Имя `qr-code` в YAML совпадает с именем папки. Каталог `skills/` передаём агенту через параметр `skills`.
Сведения об авторе и источнике находятся в поле `metadata` заголовка `SKILL.md`; скрипты распространяются по лицензии MIT.

Для декодирования нужна библиотека ZBar. В Windows x64 DLL входят в wheel pyzbar; при ошибке загрузки может понадобиться
Visual C++ Redistributable 2013 x64. На macOS используется Homebrew, в Debian/Ubuntu — libzbar0.
Инструкции по ОС приведены в README. Проверка зависимости выполняется до запуска модели.

In [ ]:
print((ROOT / "skills/qr-code/SKILL.md").read_text(encoding="utf-8"))
print("Файлы:", [str(p.relative_to(ROOT / "skills/qr-code")) for p in sorted((ROOT / "skills/qr-code").rglob("*")) if p.is_file()])

### 18.1. Предварительная проверка генератора и декодера

Используем включённые в репозиторий скрипты в отдельной рабочей папке. Этот короткий запуск позволяет убедиться,
что Python-пакеты и нативная библиотека установлены, прежде чем тратить запросы модели на агентный сценарий.
Код читает только созданный в этом примере файл.

In [ ]:
import subprocess
import shutil
import tempfile

qr_runtime = ROOT / ".runtime/qr"
qr_runtime.mkdir(parents=True, exist_ok=True)
qr_text = "https://example.com/mipt-protocols"
qr_image = qr_runtime / "check.png"
for command in [
    [sys.executable, str(ROOT / "skills/qr-code/scripts/qr_generate.py"), qr_text, str(qr_image)],
    [sys.executable, str(ROOT / "skills/qr-code/scripts/qr_read.py"), str(qr_image)],
]:
    checked = subprocess.run(command, check=True, capture_output=True, text=True, encoding="utf-8")
    print(checked.stdout)
assert checked.stdout.strip() == qr_text

### 18.2. Deep Agent выполняет навык

Параметр `skills` в `create_deep_agent` подключает штатный `SkillsMiddleware`.
Middleware — компонент, который дополняет обработку запроса до или после вызова модели.
`SkillsMiddleware` находит навыки и перед вызовом модели добавляет в контекст их имена, описания и пути к инструкциям.
Агент выбирает навык по описанию и читает его `SKILL.md` файловым инструментом `read_file`.

Временная рабочая папка содержит только выбранный навык. Backend — объект, через который агент работает
с файлами и командами. `LocalShellBackend` предоставляет файловые инструменты и исполнение shell-команд.
Для файловых инструментов и shell используем реальные пути (`virtual_mode=False`).
Каталог навыков передаём абсолютным путём, а результат сохраняем в рабочей папке.
Сам shell работает с правами Python-процесса; для неизвестных навыков требуется отдельная изолированная среда.
В данном примере используем включённые и просмотренные скрипты QR-навыка.

Агент должен прочитать SKILL.md, выполнить генератор и декодер, сопоставить строки и завершить ответ.
После него проверим результат ещё раз обычным Python-вызовом.

In [ ]:
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import MemorySaver

skill_workspace = ROOT / ".runtime/skill-workspace"
skills_dir = skill_workspace / "skills"
skills_dir.mkdir(parents=True, exist_ok=True)
copied_skill = skill_workspace / "skills/qr-code"
if copied_skill.exists():
    shutil.rmtree(copied_skill)
shutil.copytree(ROOT / "skills/qr-code", copied_skill)
# Предыдущий результат удаляется, чтобы новая проверка не приняла его за свежий.
(skill_workspace / "result.png").unlink(missing_ok=True)
skill_config = {"configurable": {"thread_id": str(__import__("uuid").uuid4())}, "recursion_limit": 50}
backend = LocalShellBackend(root_dir=skill_workspace, virtual_mode=False,
    env={"PATH": str(Path(sys.executable).parent) + os.pathsep + os.defpath})
model = make_model()
try:
    skill_agent = create_deep_agent(
        model=model, backend=backend,
        skills=[skills_dir.as_posix()], checkpointer=MemorySaver(),
        system_prompt=("Выбери подходящий навык и следуй его инструкции. "
            f"Интерпретатор Python: {sys.executable}. "
            "Сохрани result.png и сравни декодированную строку с исходной. Все зависимости уже установлены. "
            "После успешной проверки заверши ответ."),
    )
    skill_result = await skill_agent.ainvoke({"messages": [HumanMessage(content=f"Создай QR для {qr_text} и прочитай его обратно.")]},
        config=skill_config)
finally:
    await model._client.aclose()
for message in skill_result["messages"]:
    message.pretty_print()

In [ ]:
verified = subprocess.run([sys.executable, str(ROOT / "skills/qr-code/scripts/qr_read.py"), str(skill_workspace / "result.png")],
                          check=True, capture_output=True, text=True, encoding="utf-8")
assert verified.stdout.strip() == qr_text
from PIL import Image
with Image.open(skill_workspace / "result.png") as image:
    display(image.copy())
print("Независимая проверка пройдена:", verified.stdout.strip())

### 18.3. Как работает SkillsMiddleware

В предыдущем примере `create_deep_agent` создал `SkillsMiddleware` по параметру `skills`.
Middleware прочитал метаданные файлов `SKILL.md` до вызова модели. Полную инструкцию выбранного навыка
агент получил позже через `read_file`, а скрипты выполнил инструментом `execute` из `LocalShellBackend`.

`skills_dir` — папка, внутри которой находится каталог `qr-code` с `SKILL.md`.
`skills_metadata` в состоянии графа содержит обнаруженные имена, описания и пути к навыкам.
Посмотрите обнаруженный навык в состоянии графа и соответствующие вызовы в истории.
Имена и описания загружаются при обнаружении; подробная инструкция читается при выборе навыка.
[Документация SkillsMiddleware](https://docs.langchain.com/oss/python/deepagents/skills).

In [ ]:
skill_state = await skill_agent.aget_state(skill_config)
skills_metadata = skill_state.values.get("skills_metadata", [])
print("Обнаруженные навыки:", [
    {"name": item["name"], "description": item["description"]}
    for item in skills_metadata
])
assert any(item["name"] == "qr-code" for item in skills_metadata)

skill_calls = [call for message in skill_result["messages"]
               for call in getattr(message, "tool_calls", [])]
for call in skill_calls:
    if call["name"] in {"read_file", "execute"}:
        print(call["name"], call["args"])
assert any(call["name"] == "read_file" and "SKILL.md" in str(call["args"])
           for call in skill_calls)

<a id="lab-reference"></a>

## Справочник: что находится в lab.py

[`lab.py`](lab.py) содержит вспомогательный код блокнота. Он не реализует логику поиска поезда
и не содержит инструкции агента: агент поездок определён в `agents/travel.py`, серверные обработчики — в `servers/`.
Функции протокольного обмена из `lab.py` показаны выше рядом с опытами; здесь собраны их назначения и результаты.

### Настройки и подготовка

| Имя | Назначение и результат |
|---|---|
| `ROOT` | Папка проекта, относительно которой строятся пути к данным, серверам и журналам |
| Загрузка `.env` | При импорте модуль читает `.env` рядом с `lab.py`. Уже установленные переменные окружения имеют приоритет |
| `make_model(**kwargs)` | Создаёт клиент GigaChat с API v2, моделью из `GIGACHAT_MODEL`, температурой 0.01 и тайм-аутом 120 секунд |
| `future_date(days=21)` | Возвращает дату через заданное число дней в формате `YYYY-MM-DD` |

### Локальные процессы

| Имя | Назначение и результат |
|---|---|
| `LocalProcess` | Запускает дочерний процесс, сохраняет его журнал в `.runtime/<имя>.log`, предоставляет `read_log()` и `stop()` |
| `start_server(...)` | Проверяет порт, запускает модуль сервера или `langgraph dev`, ждёт HTTP-ответа и возвращает `LocalProcess` |
| `SERVERS` | Словарь процессов, запущенных этим модулем; используется для повторных запусков и остановки |
| `stop_all()` | Останавливает зарегистрированные процессы. Также вызывается при нормальном завершении Python-процесса |
| `safe_log(text)` | Скрывает значения `GIGACHAT_CREDENTIALS` и `GIGACHAT_ACCESS_TOKEN` в тексте, который возвращает `read_log()` |

Журнал можно посмотреть без повторного запроса к модели: например, `print(sdk_server.read_log())`.
Ручная остановка — `sdk_server.stop()`, общая — `lab.stop_all()`.

### MCP — § 7

| Имя | Что делает | Что возвращает |
|---|---|---|
| `mcp_request(...)` | Формирует JSON-RPC и HTTP-заголовки, отправляет запрос на `127.0.0.1:8765/mcp` | Запрос, ответ, HTTP-заголовки ответа |
| `decode_json_response(response)` | Разбирает JSON или извлекает итоговое сообщение из SSE | Словарь с JSON-RPC-ответом |

В § 4–6 используются `MCPAdapter` и клиент MCP SDK напрямую. `mcp_request` нужен для просмотра сообщений в § 7.

### A2A — § 10–12

| Имя | Что делает | Что возвращает |
|---|---|---|
| `discover_agent(base_url)` | Читает Agent Card по известному пути сервера | Словарь с карточкой |
| `platform_card()` | Находит `assistant_id` в Agent Server, затем получает его A2A-карточку | Тот же формат карточки |
| `ask_a2a(card, text)` | Отправляет `SendMessage`, проверяет завершение задачи, извлекает текст | `A2AReply(request, task, text)` |

### AG-UI и A2UI — § 13–14

| Имя | Что делает | Что возвращает |
|---|---|---|
| `agui_events(port, text)` | Отправляет запрос, читает SSE, печатает события и фрагменты ответа | Список всех полученных событий |
| `a2ui_operations(events)` | Извлекает `a2ui_operations` из результата инструмента | Список операций A2UI |
| `preview_a2ui(operations)` | Преобразует поддерживаемые компоненты в статический HTML | Строку HTML для вывода в блокноте |

### Agent Client Protocol — § 15

`ACPClient` запускает процесс агента и обменивается с ним JSON-RPC через stdin/stdout.
`request(method, params)` возвращает запрос, результат и уведомления. `on_update` показывает уведомления по мере получения;
`permission_handler` отвечает на session/request_permission; решения сохраняются в `permission_log`.
Без обработчика разрешений действия отклоняются. `stop()` завершает процесс. Файлы и тесты исполняет агент в рабочей папке.

## Словарь и источники

### MCP

- **Хост (host)** — приложение, которое создаёт MCP-клиенты и использует полученные данные и инструменты.
- **MCP-клиент** — компонент приложения для обмена сообщениями с конкретным MCP-сервером.
- **MCP-сервер** — программа, предоставляющая инструменты, ресурсы и шаблоны промптов.
- **Tool** — вызываемая операция; **resource** — данные по URI; **prompt** — шаблон сообщений с параметрами.
- **MRTR (Multi Round-Trip Requests)** — обработка запроса за несколько обращений, если серверу нужны дополнительные данные.

### LangChain и Deep Agents

- **AIMessage / ToolMessage** — классы сообщений модели и результатов инструментов в истории LangChain.
- **Граф агента** — схема выполнения шагов: вызовы модели, инструменты и переходы между ними.
- **Checkpointer** — компонент LangGraph для сохранения состояния графа между шагами и обращениями.
- **Middleware** — компонент, дополняющий обработку запроса до или после вызова модели.
- **Backend** — объект, через который агент выполняет файловые операции или команды.
- **Skill** — пакет инструкций и вспомогательных материалов для выполнения задачи.

### A2A

- **Agent Card** — описание агента: его адрес, возможности и поддерживаемые способы обмена.
- **Task** — задача с идентификатором и состоянием выполнения.
- **Artifact** — сохранённый результат задачи, например текст найденной поездки.

### Интерфейс приложения

- **Событие AG-UI** — сообщение о ходе работы агента, например начале поиска или очередном фрагменте ответа.
- **Рендерер A2UI** — код приложения, который строит интерфейс по описанию компонентов.
- **Сессия Agent Client Protocol** — диалог приложения с агентом, объединяющий запросы и ответы.

### Форматы данных и обмен

- **URI** — идентификатор ресурса, например `course://overview`.
- **JSON** — текстовый формат данных: объекты с полями, списки и простые значения.
- **JSON Schema** — описание структуры данных: имён полей, типов, обязательности и допустимых значений.
- **JSON-RPC** — формат запроса метода с аргументами и `id`; ответ содержит тот же `id` и результат или ошибку.
- **stdin / stdout / stderr** — потоки ввода, вывода и диагностики процесса.
- **SSE (Server-Sent Events)** — передача сообщений от сервера потоком внутри HTTP-ответа.
- **OAuth** — получение разрешения пользователя на доступ приложения к данным сервиса.
- **PKCE** — защита обмена кода авторизации на токен доступа.
- **Токен доступа** — значение, которое приложение предъявляет сервису в подтверждение разрешённого доступа.
- **top-k** — первые k результатов поиска в порядке релевантности.

Основные первоисточники:
[MCP](https://modelcontextprotocol.io/specification/2026-07-28),
[MCP Python SDK](https://py.sdk.modelcontextprotocol.io/),
[LangChain MCP](https://docs.langchain.com/oss/python/langchain/mcp),
[A2A](https://a2a-protocol.org/latest/specification/),
[AG-UI](https://docs.ag-ui.com/), [A2UI](https://a2ui.org/),
[Agent Client Protocol](https://agentclientprotocol.com/get-started/introduction),
[Skills Deep Agents](https://docs.langchain.com/oss/python/deepagents/skills).

In [ ]:
lab.stop_all()
print("Работа завершена; локальные серверы остановлены")